# Introduction to AutoGen: Credit Risk Assessment Workflow

This workflow follows a realistic credit-risk process:

- Document verification for registration and GST evidence
- Compliance checks such as MCA and sanctions review
- Financial analysis using the latest available year data
- Credit bureau review when score or rating data is available
- Industry-risk classification using the provided matrix
- Final risk assessment using a 100-point scorecard covering Compliance (20), Liquidity (20), Leverage (15), Profitability (15), External Credit Bureau (15), and Industry Risk (15)

The final score is used to determine the applicable risk category, credit limit, and payment terms according to the provided assessment rules.

## Install dependencies


In [ ]:
%pip install -qU autogen "autogen-agentchat" "autogen-ext[openai,azure]"

### Initialising Azure OpenAI client to invoke the LLM


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from autogen_core.models import UserMessage

Please fill in `api_key="<replace_your_api_key>"` by replacing `<replace_your_api_key>` with your actual API key.

> The Azure configuration values already provided for `azure_deployment`, `model`, `api_version`, and `azure_endpoint` can stay as they are for this lab.


In [ ]:
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

az_model_client = AzureOpenAIChatCompletionClient(
    azure_deployment="gpt-5.4-mini",
    model="gpt-5.4-mini",
    api_version="2025-01-01-preview",
    azure_endpoint="https://hakunamatata11.openai.azure.com/",
    api_key="<replace_your_api_key>",
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
        "structured_output": True,
    },
)

print("Azure OpenAI model client ready")

## Logging


In [ ]:
import logging

from autogen_core import EVENT_LOGGER_NAME

# Configure the root logger so events are visible when debugging agent behavior.
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(EVENT_LOGGER_NAME)
logger.addHandler(logging.StreamHandler())
logger.setLevel(logging.INFO)


AutoGen uses the standard Python logging module to emit events such as model calls, tool executions, and responses. Logging is useful because it helps you understand what an agent is doing behind the scenes while debugging or monitoring a workflow. In practice, it shows when the model is called, when tools are invoked, and how the agent is progressing through a task.


## Message System


AutoGen agents communicate through messages. In this credit-risk workflow, messages carry details such as company documents, compliance status, financial ratios, and assessment decisions. This message-based approach is important because each component of the workflow can share structured information in a controlled and traceable way.

Messages generally fall into two categories:

- Agent-to-agent messages: text, media, and structured payloads
- Internal events: tool calls, status updates, and workflow notifications


### Agent-Agent Messages

Agent-to-agent messages inherit from `BaseChatMessage`. The most common examples are:

- `TextMessage` for plain text
- `MultiModalMessage` for document or image content


In [ ]:
from autogen_agentchat.messages import TextMessage

# A basic text message represents a user request or a message passed between agents.
text_message = TextMessage(
    content="Review the credit-risk application for Apex Manufacturing Pvt Ltd.",
    source="relationship_manager",
)
text_message


### Multi Modal messages


In [ ]:
from PIL import Image
from autogen_agentchat.messages import MultiModalMessage
from autogen_core import Image as AGImage

# Load the sample company registration document.
pil_image = Image.open("../../media/company_registration_certificate.png")
img = AGImage(pil_image)

multi_modal_message = MultiModalMessage(
    content=[
        "Review the company registration document and identify the company name shown in the document.",
        img,
    ],
    source="credit_ops",
)

img

# Test the multimodal agent 
By providing the company registration certificate image and asking the agent to identify the company name and company number.

In [ ]:
from autogen_agentchat.agents import AssistantAgent

multimodal_agent = AssistantAgent(
    name="document_verification_agent",
    model_client=az_model_client,
    system_message=(
        "You are a credit-risk document verification agent. "
        "Review the provided company registration document and identify "
        "the company name and company number visible in the document."
    ),
    model_client_stream=False,
)

result = await multimodal_agent.run(task=multi_modal_message)

print(result.messages[-1].content)

### Subclasses in messages

Why and when to subclass?

AutoGen provides abstract base classes for both message exchange and internal workflow events. Subclassing is useful when you want to attach structured business data, override the text representation, or pass additional metadata to a user interface or monitoring layer. For example, in a credit-risk workflow, a custom message can carry document status, sanctions flags, and scoring information in a structured format.


Imports & helper models


In [ ]:
from __future__ import annotations

from typing import Any, Dict
from pydantic import BaseModel
from autogen_agentchat.messages import (
    BaseChatMessage,
    BaseAgentEvent,
)
from autogen_core.models import UserMessage

### Custom Chat Message (`CreditRiskContextMessage`)

This message carries structured credit-risk context such as company name, document status, and compliance notes.


In [ ]:
class CreditRiskContextMessage(BaseChatMessage):
    """
    A message that carries structured data for a credit-risk application.
    This is useful when different agents need to pass business context in a consistent format.
    """

    content: Dict[str, Any]

    def to_text(self) -> str:
        return f"[Credit Risk Context] {self.content}"

    def to_model_text(self) -> str:
        return str(self.content)

    def to_model_message(self) -> UserMessage:
        return UserMessage(content=self.to_model_text(), source=self.source)


### Custom Internal Event (`RiskAlertEvent`)

This event represents a workflow alert such as a missing document or a compliance exception.


In [ ]:
class RiskAlertEvent(BaseAgentEvent):
    """
    An internal event signalling that a credit-risk validation issue was raised.
    This is useful for alerts, warnings, or operational signals that should not be sent directly to the model.
    """

    level: str
    details: str

    def to_text(self) -> str:
        return f"[{self.level}] {self.details}"


### Minimal Demo with a Credit Risk Analyst Agent

This example shows how a single `AssistantAgent` can review a credit application, check documents, and produce a structured result.


In [ ]:
pip install nest-asyncio

In [ ]:
from autogen_core import CancellationToken
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
import nest_asyncio

nest_asyncio.apply()

async def verify_document_pack(company_name: str, registration_certificate: str, gst_certificate: str) -> str:
    """Check whether the required company documents are available and whether the names match."""
    if not registration_certificate or not gst_certificate:
        return "Missing required documents: registration certificate and/or GST certificate."
    if company_name.lower() not in registration_certificate.lower() or company_name.lower() not in gst_certificate.lower():
        return "Company name mismatch across documents."
    return "Documents available and company names match."

credit_risk_agent = AssistantAgent(
    name="credit_risk_analyst",
    model_client=az_model_client,
    tools=[verify_document_pack],
    system_message="You are the credit risk analyst. Review the validation request and clearly flag missing documents or name mismatches.",
    reflect_on_tool_use=True,
    model_client_stream=False,
)

async def run_credit_risk_demo():
    token = CancellationToken()
    response = await credit_risk_agent.on_messages(
        [TextMessage(content="Check the document pack for Apex Manufacturing Pvt Ltd. Registration certificate is available and names match. GST certificate is missing.", source="relationship_manager")],
        cancellation_token=token,
    )
    print(response.chat_message.content)

await run_credit_risk_demo()

With these patterns, you can build structured, auditable interactions for a real credit-risk workflow. This design helps keep the process organized, explainable, and easier to extend as the business logic grows.


## Understanding agents


AutoGen AgentChat includes built-in agents designed for different tasks. In a credit-risk workflow, the same principles apply: each agent preserves state, accepts messages, and may use tools to perform structured checks.

`name`: A unique identifier for the agent.

`description`: A summary of the agent’s role.

`run(task)`: Executes the agent on a new task and returns a `TaskResult`.

`run_stream(task)`: Returns an asynchronous stream of messages, useful for processing status updates.

These agents rely on the message classes from `autogen_agentchat.messages`.


In [ ]:
async def get_compliance_status(company_name: str, mca_status: str | None = None, sanctions_status: str | None = None, adverse_record_status: str | None = None) -> str:
    """Return the current compliance status and clearly mark unavailable checks."""
    status_parts = [f"Company: {company_name}"]
    if mca_status is None:
        status_parts.append("MCA status unavailable.")
    else:
        status_parts.append(f"MCA status: {mca_status}")
    if sanctions_status is None:
        status_parts.append("Sanctions check unavailable.")
    else:
        status_parts.append(f"Sanctions check: {sanctions_status}")
    if adverse_record_status is None:
        status_parts.append("Adverse record check unavailable.")
    else:
        status_parts.append(f"Adverse record check: {adverse_record_status}")
    return "; ".join(status_parts)

# This helper computes a few core financial indicators that are commonly used in credit review.
async def calculate_financial_ratios(current_assets: float, current_liabilities: float, total_debt: float, total_equity: float, revenue: float, net_income: float) -> str:
    """Compute key ratios for the latest financial year using deterministic logic."""
    current_ratio = current_assets / current_liabilities if current_liabilities else None
    debt_to_equity = total_debt / total_equity if total_equity else None
    net_profit_margin = net_income / revenue if revenue else None
    return (
        f"Current Ratio={current_ratio}, "
        f"Debt-to-Equity={debt_to_equity}, "
        f"Net Profit Margin={net_profit_margin}"
    )

# This function represents external bureau information, which may be missing in some applications.
async def get_credit_bureau_summary(score: str | None = None, rating: str | None = None) -> str:
    """Return bureau information if available; otherwise flag it as missing."""
    if score is None and rating is None:
        return "External credit bureau information unavailable."
    return f"Bureau score={score}, bureau rating={rating}"


In [ ]:
credit_analyst = AssistantAgent(
    name="credit_analyst",
    model_client=az_model_client,
    tools=[get_compliance_status, calculate_financial_ratios, get_credit_bureau_summary],
    system_message="You are a credit-risk analyst responsible for interpreting company documents, compliance signals, financial ratios, and bureau information.",
    reflect_on_tool_use=True,
    model_client_stream=False,
)

Getting responses
We can use the `run()` method to execute a credit-risk assessment request. When you call `run()`, the method returns a `TaskResult` object with a `messages` attribute that captures the agent's reasoning and final reply.

The important part is that the agent stores internal context between runs. For a credit-risk workflow, that means you should pass only the new request or a focused summary instead of resending the entire assessment history each time.

`run()` → `TaskResult` → `messages` list shows the workflow steps and final answer.


In [ ]:
async def assess_credit_application() -> None:
    result = await credit_analyst.run(
        task="Assess the credit-risk application for Apex Manufacturing Pvt Ltd using the latest available compliance, financial, and bureau inputs."
    )
    print(result.messages[-1].content)

await assess_credit_application()

### Multi-Modal Input
The `AssistantAgent` can also review a certificate image as part of a document verification workflow.


In [ ]:
from io import BytesIO

import PIL
import requests
from autogen_agentchat.messages import MultiModalMessage
from autogen_core import Image

response = requests.get("https://picsum.photos/300/200")
response.raise_for_status()
pil_image = PIL.Image.open(BytesIO(response.content))
img = Image(pil_image)
multimodal_message = MultiModalMessage(
    content=[
        "Review this certificate image and tell me whether the company name and registration status look valid for a credit application.",
        img,
    ],
    source="credit_ops",
)
img

multimodal_agent = AssistantAgent(
    name="document_verification_agent",
    model_client=az_model_client,
    system_message="You are a document verification agent. Check the certificate for name consistency and missing legal information.",
    reflect_on_tool_use=False,
    model_client_stream=False,
)

result = await multimodal_agent.run(task=multimodal_message)
print(result.messages[-1].content)

### Streaming Messages

The `run_stream()` method allows a credit-risk workflow to emit updates as each stage is processed. This is especially useful when a workflow is checking documents, compliance, financials, and bureau data in sequence.

It returns an asynchronous generator. In this demo, the Model client is kept non-streaming to avoid the Azure preview issue while preserving the same AutoGen pattern and workflow logic.


In [ ]:
from autogen_agentchat.messages import TextMessage

streaming_agent = AssistantAgent(
    name="risk_streaming_assistant",
    model_client=az_model_client,
    system_message="You are a credit-risk workflow assistant. Report status updates as the document, compliance, financial, and bureau checks proceed.",
    model_client_stream=False,
)

async def credit_stream_task():
    print("\nCredit-risk workflow output:\n")
    async for msg in streaming_agent.run_stream(task="Perform a full credit-risk review for Apex Manufacturing Pvt Ltd and report processing steps."):
        if hasattr(msg, "type") and hasattr(msg, "content"):
            print(f"{msg.type}: {msg.content}")
        elif isinstance(msg, TextMessage):
            print(f"TextMessage: {msg.content}")
        else:
            print(f"[TaskResult] Task completed with {len(msg.messages)} messages.")

await credit_stream_task()

## Using tools with agents


### Why Add Tools to LLMs?

Credit-risk reviews need more than language generation. They need deterministic checks for:

* legal document verification
* compliance lookups
* ratio calculations
* bureau score retrieval
* industry-risk classification

Tool calling lets the LLM request structured outputs from deterministic logic and then reason over them safely.


### FunctionTool: Turning a Python Function into a Credit-Risk Tool

Any `async` Python function with type hints and a clear docstring can be registered as an AutoGen tool. In a credit-risk workflow, this is ideal for deterministic checks and financial calculations.


In [ ]:
from autogen_core.tools import FunctionTool

async def document_lookup(company_name: str) -> str:
    """Return the latest document status for a company."""
    return f"Document check for {company_name}: Company Registration Certificate present; GST Certificate present; company name matches."

credit_document_tool = FunctionTool(document_lookup, description="Check whether required documents are present and valid for a credit application.")
credit_document_tool.schema

### AssistantAgent + Reflection

`AssistantAgent` is a good fit for a credit-risk analyst because it can use tools and then explain the result in plain English without exposing the implementation details.


In [ ]:
nest_asyncio.apply()

credit_assistant = AssistantAgent(
    name="credit_assistant",
    model_client=az_model_client,
    tools=[document_lookup, get_compliance_status, calculate_financial_ratios],
    system_message="Use the available tools to assess the credit application. Clearly note missing information and show the calculations used.",
    reflect_on_tool_use=True,
    model_client_stream=False,
)

async def run_credit_assistant_demo():
    await Console(
        credit_assistant.run_stream(
            task="Assess the credit-risk file for Apex Manufacturing Pvt Ltd using document, compliance, and financial inputs."
        )
    )

await run_credit_assistant_demo()

### Parallel Tool Calls

Some model backends can propose multiple tool calls in a single response. In a credit-risk workflow, that means a single agent can inspect documents, compliance, and financial ratios concurrently when the checks are independent.

If there are shared side effects or ordering requirements, disable parallel calls and keep deterministic execution.


## Structured Output in AutoGen


* Goal: receive a credit-risk assessment as a validated Python object instead of raw text.
* How: define a `pydantic.BaseModel` and pass it to `AssistantAgent(..., output_content_type=YourModel)`.
* Result: the agent returns a structured message containing a validated model instance.
* This is useful for auditability, downstream workflows, and rule-based credit decisions.


In [ ]:
%pip install pydantic --quiet

### Define the Pydantic Schema


In [ ]:
from typing import Literal
from pydantic import BaseModel

class CreditRiskResult(BaseModel):
    company_name: str
    document_status: str
    compliance_status: str
    financial_summary: str
    bureau_status: str
    industry_risk: str
    compliance_score: float
    liquidity_score: float
    leverage_score: float
    profitability_score: float
    bureau_score: float
    industry_risk_score: float
    final_score: float
    risk_assessment: str
    credit_limit_recommendation: str
    payment_terms_recommendation: str

### Instantiate the credit-risk agent


In [ ]:
from autogen_agentchat.messages import StructuredMessage

credit_assessment_agent = AssistantAgent(
    name="credit_assessment_agent",
    model_client=az_model_client,
    system_message="Assess the client using the provided scorecard, document checks, compliance reviews, financial ratios, bureau status, and industry risk matrix.",
    output_content_type=CreditRiskResult,
    model_client_stream=False,
)

### Run and validate


In [ ]:
async def assess_structured_credit_risk():
    result = await credit_assessment_agent.run(
        task=(
            "Assess Apex Manufacturing Pvt Ltd. Use the document verification results, compliance status, latest financial-year ratios, "
            "latest bureau information if available, and the industry-risk matrix. Show the score breakdown and final assessment."
        )
    )

    if not result.messages:
        raise ValueError("The model did not return any messages for the structured credit-risk assessment.")

    last_msg = result.messages[-1]
    structured_content = None

    if isinstance(last_msg, StructuredMessage):
        structured_content = last_msg.content
    elif isinstance(getattr(last_msg, "content", None), dict):
        structured_content = CreditRiskResult.model_validate(last_msg.content)
    else:
        print("Structured output was not returned; the model replied with plain text instead.")
        print(getattr(last_msg, "content", str(last_msg)))
        return

    if not isinstance(structured_content, CreditRiskResult):
        try:
            structured_content = CreditRiskResult.model_validate(structured_content)
        except Exception:
            print("Could not validate the structured response from the model.")
            print(structured_content)
            return

    print("Final score:", structured_content.final_score)
    print("Score breakdown:", {
        "compliance": structured_content.compliance_score,
        "liquidity": structured_content.liquidity_score,
        "leverage": structured_content.leverage_score,
        "profitability": structured_content.profitability_score,
        "bureau": structured_content.bureau_score,
        "industry": structured_content.industry_risk_score,
    })
    print("Risk assessment:", structured_content.risk_assessment)
    print("Credit limit recommendation:", structured_content.credit_limit_recommendation)
    print("Payment terms recommendation:", structured_content.payment_terms_recommendation)

await assess_structured_credit_risk()


## Controlling the Model's Context Window


The default context behavior can accumulate the full credit-risk assessment history. For a workflow that spans documents, compliance, and financial analysis, it is often useful to retain only the most relevant information in the current assessment.

This is where a buffered context helps. It keeps the latest messages in memory and avoids passing the entire history on every step.


### Setting up the agent with `BufferedChatCompletionContext`


In [ ]:
from autogen_core.model_context import BufferedChatCompletionContext

async def credit_context_lookup(query: str) -> str:
    return f"Latest assessment context for {query}: company documents validated, compliance status available, bureau status missing, industry risk medium."

credit_context_agent = AssistantAgent(
    name="credit_context_agent",
    model_client=az_model_client,
    tools=[credit_context_lookup],
    system_message="Keep the current credit-risk assessment context concise and relevant.",
    model_context=BufferedChatCompletionContext(buffer_size=5),
    model_client_stream=False,
)

### Preview in action


In [ ]:
async def run_credit_context_demo():
    await Console(
        credit_context_agent.run_stream(
            task="Continue the credit-risk assessment for Apex Manufacturing Pvt Ltd using only the latest relevant context."
        )
    )

await run_credit_context_demo()

## Saving and Loading Agent State in AutoGen


A credit-risk workflow can be stateful. The assessment may span several steps and intermediate findings. AutoGen allows you to save the agent state and resume later with the same context.

This is especially useful if a credit analyst pauses and resumes an application review.


### Create and run a credit-risk assessment, then save state


In [ ]:
import json

assessment_agent = AssistantAgent(
    name="assessment_agent",
    system_message="You are the credit-risk analyst. Maintain the current customer assessment context.",
    model_client=az_model_client,
    model_client_stream=False,
)

response = await assessment_agent.on_messages(
    [TextMessage(content="Prepare a summary for the credit-risk file for Apex Manufacturing Pvt Ltd. Document pack is complete and company names match.", source="credit_ops")],
    cancellation_token=CancellationToken(),
)

print(response.chat_message.content)

assessment_state = await assessment_agent.save_state()
with open("credit_risk_assessment_state.json", "w") as f:
    json.dump(assessment_state, f)


### Load the saved state into a new assessment agent


In [ ]:
with open("credit_risk_assessment_state.json", "r") as f:
    saved_state = json.load(f)

new_assessment_agent = AssistantAgent(
    name="assessment_agent",
    system_message="You are the credit-risk analyst. Continue the same customer assessment context.",
    model_client=az_model_client,
    model_client_stream=False,
)

await new_assessment_agent.load_state(saved_state)

resume_response = await new_assessment_agent.on_messages(
    [TextMessage(content="What was the latest assessment status we captured for this customer?", source="credit_ops")],
    cancellation_token=CancellationToken(),
)

print(resume_response.chat_message.content)

## Building and Using Custom Agents


AutoGen allows you to define specialized agents for the discrete steps in a credit-risk workflow. This is useful when each agent has a clear responsibility: document verification, compliance review, financial analysis, bureau validation, or final assessment synthesis.

Each custom agent inherits from `BaseChatAgent` and implements the required methods for messages and resets.


### Document Verification Agent

This custom agent reviews whether required documents are available and whether company names match across those documents.


In [ ]:
from typing import AsyncGenerator, Sequence
from autogen_agentchat.agents import BaseChatAgent
from autogen_agentchat.base import Response
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage, TextMessage

class DocumentVerificationAgent(BaseChatAgent):
    def __init__(self, name: str):
        super().__init__(name, "Verifies company registration and GST documentation.")

    @property
    def produced_message_types(self) -> Sequence[type[BaseChatMessage]]:
        return (TextMessage,)

    async def on_messages(self, messages: Sequence[BaseChatMessage], cancellation_token: CancellationToken) -> Response:
        content = ""
        for message in messages:
            content += str(message.content) + "\n"
        status = "Document pack verified: Company Registration Certificate is available; GST Certificate is available; company names match." if "Apex Manufacturing Pvt Ltd" in content else "Document pack issue: one or more required documents are missing or company names do not match."
        return Response(chat_message=TextMessage(content=status, source=self.name))

    async def on_messages_stream(self, messages: Sequence[BaseChatMessage], cancellation_token: CancellationToken) -> AsyncGenerator[BaseAgentEvent | BaseChatMessage | Response, None]:
        yield TextMessage(content="Checking company registration and GST documents...", source=self.name)
        yield Response(chat_message=TextMessage(content="Document verification complete.", source=self.name))

    async def on_reset(self, cancellation_token: CancellationToken) -> None:
        pass

verification_agent = DocumentVerificationAgent("document_verification")
async def run_document_verification_agent():
    async for msg in verification_agent.on_messages_stream([], CancellationToken()):
        print(msg.content if hasattr(msg, "content") else msg)

await run_document_verification_agent()


### Financial Analysis Agent

This custom agent applies deterministic financial calculations for the latest available year.


In [ ]:
class FinancialAnalysisAgent(BaseChatAgent):
    def __init__(self, name: str):
        super().__init__(name, "Calculates liquidity, leverage, and profitability metrics.")

    @property
    def produced_message_types(self) -> Sequence[type[BaseChatMessage]]:
        return (TextMessage,)

    async def on_messages(self, messages: Sequence[BaseChatMessage], cancellation_token: CancellationToken) -> Response:
        current_assets = 2_000_000
        current_liabilities = 1_000_000
        total_debt = 600_000
        total_equity = 1_500_000
        revenue = 4_000_000
        net_income = 340_000

        current_ratio = current_assets / current_liabilities
        debt_to_equity = total_debt / total_equity
        net_profit_margin = net_income / revenue
        summary = (
            f"Current Ratio={current_ratio:.2f}; "
            f"Debt-to-Equity={debt_to_equity:.2f}; "
            f"Net Profit Margin={net_profit_margin:.2f}"
        )
        return Response(chat_message=TextMessage(content=summary, source=self.name))

    async def on_messages_stream(self, messages: Sequence[BaseChatMessage], cancellation_token: CancellationToken) -> AsyncGenerator[BaseAgentEvent | BaseChatMessage | Response, None]:
        yield TextMessage(content="Calculating financial ratios for the latest FY...", source=self.name)
        yield Response(chat_message=TextMessage(content="Financial analysis complete.", source=self.name))

    async def on_reset(self, cancellation_token: CancellationToken) -> None:
        pass

financial_agent = FinancialAnalysisAgent("financial_analysis")
async def run_financial_agent():
    async for msg in financial_agent.on_messages_stream([], CancellationToken()):
        print(msg.content if hasattr(msg, "content") else msg)

await run_financial_agent()

## Credit Risk Team and Selector

This final section shows how multiple specialist agents collaborate on the same credit-risk assessment. The team routes each part of the workflow to the most relevant agent and then synthesizes the final assessment. In a real lending operation, this allows distinct responsibilities such as document review, financial analysis, and final scoring to be handled by specialized components while remaining connected through a single coordinated workflow.


In [ ]:
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import SelectorGroupChat

credit_specialists = [
    DocumentVerificationAgent("document_verification"),
    FinancialAnalysisAgent("financial_analysis"),
    credit_analyst,
]

team = SelectorGroupChat(
    credit_specialists,
    model_client=az_model_client,
    termination_condition=MaxMessageTermination(8),
    allow_repeated_speaker=True,
    selector_prompt=(
        "Available roles:\n{roles}\nTheir job descriptions:\n{participants}\n"
        "Current workflow:\n{history}\nChoose the most appropriate specialist for the next credit-risk step."
    ),
)

async def run_credit_team():
    task = [
        TextMessage(content="Review the credit-risk application for Apex Manufacturing Pvt Ltd. Validate the document pack, confirm compliance status, compute financial ratios, check bureau status, assess industry risk, and prepare the final scorecard.", source="relationship_manager"),
        TextMessage(content="Apex Manufacturing Pvt Ltd", source="relationship_manager"),
    ]
    await Console(team.run_stream(task=task))

await run_credit_team()


> **Note:** The AutoGen team output may be longer than the earlier examples. This is expected because the workflow involves multiple specialized agents, tool calls, and intermediate messages. Focus on the **final `chat_message` response** at the end, which contains the consolidated assessment.

# On running the last cell, you can see the generated `credit_risk_assessment_state.json` file containing the saved state of the credit-risk assessment agent:

```json
{
  "type": "AssistantAgentState",
  "version": "1.0.0",
  "llm_context": {
    "messages": [
      {
        "content": "Prepare a summary for the credit-risk file for Apex Manufacturing Pvt Ltd. Document pack is complete and company names match.",
        "source": "credit_ops",
        "type": "UserMessage"
      },
      {
        "content": "Apex Manufacturing Pvt Ltd — Credit-Risk File Summary\n\n- Document pack reviewed: complete\n- Company name verification: consistent across documents; no mismatches noted\n- Preliminary file status: administratively in order for credit review\n\nAssessment note:\n- No documentary discrepancies identified at this stage based on the information provided.\n\nRecommended next step:\n- Proceed with standard credit underwriting analysis, including financial review, KYC/ownership checks, and exposure assessment.",
        "thought": null,
        "source": "assessment_agent",
        "type": "AssistantMessage"
      }
    ]
  }
}

### Conclusion

This notebook demonstrates how AutoGen can be used to build and coordinate a practical credit-risk assessment workflow. By combining messages, tools, structured outputs, context control, state management, and specialized agents, we create a system that is both extensible and grounded in real business logic. The result is a clear example of how multi-agent orchestration can support a more transparent, modular, and scalable decision process for credit evaluation.
